# DuckLake Basics

Демонстрация базовых возможностей DuckLake:
- Подключение к DuckLake через PostgreSQL catalog + MinIO
- CRUD-операции
- Просмотр снэпшотов
- Статистика таблиц

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import duckdb
import pandas as pd

# Для локального запуска вне Docker
os.environ.setdefault('DUCKLAKE_PG_HOST', 'localhost')
os.environ.setdefault('DUCKLAKE_PG_DB', 'ducklake_catalog')
os.environ.setdefault('DUCKLAKE_PG_USER', 'ducklake')
os.environ.setdefault('DUCKLAKE_PG_PASSWORD', 'ducklake_secret_change_me')
os.environ.setdefault('MINIO_ACCESS_KEY', 'minioadmin')
os.environ.setdefault('MINIO_SECRET_KEY', 'minioadmin')
os.environ.setdefault('MINIO_ENDPOINT', 'http://localhost:9000')
os.environ.setdefault('MINIO_BUCKET', 'ducklake-flights')

from src.generators.connection import get_ducklake_connection
conn = get_ducklake_connection(read_only=True)
print('Connected to DuckLake!')

## Статистика таблиц

In [ ]:
tables = ['airports', 'airlines', 'routes', 'flights', 'bookings', 'passengers', 'price_history']

stats = []
for t in tables:
    try:
        count = conn.execute(f'SELECT COUNT(*) FROM flights.{t}').fetchone()[0]
        stats.append({'table': t, 'rows': count})
    except Exception as e:
        stats.append({'table': t, 'rows': f'ERROR: {e}'})

pd.DataFrame(stats)

## Аэропорты РФ

In [ ]:
df = conn.execute("""
    SELECT iata_code, name, city, latitude, longitude
    FROM flights.airports
    ORDER BY city
""").df()
print(f'Аэропортов РФ: {len(df)}')
df.head(15)

## Маршруты по авиакомпаниям

In [ ]:
df = conn.execute("""
    SELECT
        r.airline_iata,
        al.name AS airline_name,
        COUNT(*) AS routes_count
    FROM flights.routes r
    LEFT JOIN flights.airlines al ON r.airline_iata = al.iata_code
    GROUP BY 1, 2
    ORDER BY routes_count DESC
""").df()
df

## Последние рейсы

In [ ]:
df = conn.execute("""
    SELECT
        flight_number,
        airline_iata,
        src_airport_iata || ' → ' || dst_airport_iata AS route,
        flight_date,
        status,
        total_seats
    FROM flights.flights
    ORDER BY created_at DESC
    LIMIT 20
""").df()
df

## Распределение бронирований по классам

In [ ]:
df = conn.execute("""
    SELECT
        fare_class,
        COUNT(*) AS bookings,
        ROUND(AVG(price_rub), 2) AS avg_price,
        ROUND(SUM(price_rub), 2) AS total_revenue
    FROM flights.bookings
    GROUP BY fare_class
    ORDER BY bookings DESC
""").df()
df

## Снэпшоты DuckLake

In [ ]:
# DuckLake создаёт снэпшот при каждой транзакции
# Это основа для time travel
df = conn.execute("""
    SELECT snapshot_id, snapshot_time, schema_version
    FROM ducklake_snapshots('flights')
    ORDER BY snapshot_id DESC
    LIMIT 10
""").df()
print(f'Всего снэпшотов: {len(df)}')
df

In [ ]:
conn.close()
print('Done.')